# 00 · Start here: source coverage and reproducibility

This notebook checks the evidence actually bundled with AEGIS. It **rebuilds the data dictionary**, verifies available snapshot checksums, and displays recorded coverage and limitations. It does not refresh a provider or train a model.

Run **Kernel → Restart Kernel and Run All Cells** with the documented Python 3.13 environment. Every notebook creates its own isolated workspace under `reports/notebook_runs/`; it can be run independently. Notebook 01 reconstructs preprocessing/integration, notebook 02 reruns model fitting/evaluation, and notebook 03 reruns scenario analysis.

The small **processed evidence snapshots and five selected NOAA responses are bundled**. The original bulk FSIS/CDC/USDA downloads are not. The original-file inventory below is a historical audit, not a claim that the judge ZIP contains those files. The optional original-data route is explained in notebook 01.


In [1]:
from pathlib import Path
import hashlib
import json
import sys

import pandas as pd
from IPython.display import display, Markdown

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "scripts/train_integrated_study.py").is_file() and (p / "backend").is_dir()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.notebook_support import NotebookRun

run = NotebookRun(ROOT, "00_coverage")
print("Source project:", ROOT)
print("Isolated notebook workspace:", run.workspace)
print("Original datasets, published reports and operator assignments are not notebook output targets.")


Source project: C:\Users\ericp\Downloads\NAFSI
Isolated notebook workspace: C:\Users\ericp\Downloads\NAFSI\reports\notebook_runs\00_coverage-8f791e98-f991-4a56-8b1e-b990cc3896a6\project
Original datasets, published reports and operator assignments are not notebook output targets.


## 1. Keep the evidence layers distinct

FSIS selected laboratory samples provide the food/animal outcome. CDC BEAM human-isolate aggregates and NORS outbreak records retain separate observational units. NOAA station measurements describe historical ambient conditions. Simulated logistics and cargo replay describe a fictional distributor.

Reproducing a calculation does not establish prospective validity, causal attribution, connected hardware, or regulatory authority. The owner disclosed that Kiro was not used; that challenge criterion remains unmet.


In [2]:
labels = pd.DataFrame([
    {"layer": "FSIS poultry sampling", "provenance": "Observed selected primary laboratory samples", "permitted interpretation": "Selected-sample Salmonella positivity; not consumer illness or lot clearance"},
    {"layer": "CDC BEAM", "provenance": "PulseNet-derived aggregate surveillance context", "permitted interpretation": "Human-isolate context; no genomic or individual-FSIS linkage"},
    {"layer": "CDC NORS", "provenance": "Historical exported outbreak records", "permitted interpretation": "Reported outbreak context; not additional FSIS outcome labels"},
    {"layer": "NOAA/NCEI", "provenance": "Observed historical ambient weather", "permitted interpretation": "Five selected airport stations; not cargo exposure"},
    {"layer": "Forecast/alert fixtures", "provenance": "Separately dated public-provider snapshots", "permitted interpretation": "Forecast or alert as retrieved then; not a fresh live request"},
    {"layer": "Logistics and cargo telemetry", "provenance": "Deterministic simulation", "permitted interpretation": "Scenario assumptions and quality estimates; no real shipment or hardware claim"},
])
display(labels)


,layer,provenance,permitted interpretation
0,FSIS poultry sampling,Observed selected primary laboratory samples,Selected-sample Salmonella positivity; not con...
1,CDC BEAM,PulseNet-derived aggregate surveillance context,Human-isolate context; no genomic or individua...
2,CDC NORS,Historical exported outbreak records,Reported outbreak context; not additional FSIS...
3,NOAA/NCEI,Observed historical ambient weather,Five selected airport stations; not cargo expo...
4,Forecast/alert fixtures,Separately dated public-provider snapshots,Forecast or alert as retrieved then; not a fre...
5,Logistics and cargo telemetry,Deterministic simulation,Scenario assumptions and quality estimates; no...


## 2. Inspect the historical original-file inventory

The audit records the original snapshot names, sizes, hashes, recognition and read errors. Its stored workstation paths are provenance, not locations to follow on a judge's computer. Original files are not opened by this cell. The audit's row counts/missingness are bounded or sampled where its policy says so.


In [3]:
inventory = run.read_json("data/manifests/inventory.json")
print("Recorded audit timestamp:", inventory.get("retrieved_at", "not recorded"))
print("Audit sampling policy:", inventory.get("sampling_policy", "not recorded"))
inventory_rows = pd.DataFrame([
    {
        "filename": entry["filename"],
        "bytes": entry.get("size_bytes"),
        "format": entry.get("format"),
        "content_recognition": entry.get("likely_source", "not identified"),
        "duplicate_of": entry.get("duplicate_of"),
        "recorded_errors": "; ".join(map(str, entry.get("errors", []))),
        "recorded_sha256": entry.get("sha256"),
    }
    for entry in inventory["files"]
])
display(inventory_rows)
display(pd.DataFrame([{
    "inventoried_original_files": len(inventory_rows),
    "total_inventoried_bytes": int(inventory_rows["bytes"].fillna(0).sum()),
    "files_with_recorded_read_errors": int(inventory_rows["recorded_errors"].ne("").sum()),
    "scope": "Recorded original inventory; not a fresh original-data audit",
}]))


Recorded audit timestamp: 2026-10-27T20:59:15.270672+00:00
Audit sampling policy: 200 records; workbook 201 rows/sheet; JSON fully streamed for syntax and object counts. Missingness only for sampled CSV/JSONL records.


,filename,bytes,format,content_recognition,duplicate_of,recorded_errors,recorded_sha256
0,2025-food-environment-atlas-data.xlsx,9104656,.xlsx,USDA Food Environment Atlas,None,,cd40db3986b1efca46b8d06cb0c5fdec2bb6e9f1ae697a...
1,2025-sram-fara-data.xlsx,112608645,.xlsx,USDA food access atlas,None,,fd0eff35de6664e27bb5ecbffbe1e8e5ea044979d4237b...
2,AEGIS API & Dataset Registry.xlsx,8266,.xlsx,Unrecognized / documentation; do not merge,None,,93a700c02f0a47d7b7ce4659b75ab28fe00e1398ee1e9e...
3,BEAM_Dashboard_-_Isolates_by_HHS_Region_202610...,100469,.csv,Possible CDC BEAM; requires metadata confirmation,None,,c6c4581451f87a4b34b19b38d205cc33a4b8a156149263...
4,BEAM_Dashboard_-_Report_Data_20261026.csv,17585042,.csv,Possible CDC BEAM; requires metadata confirmation,None,,c785791a096985c19c3b2a20c46112e629ae0dc73361b9...
5,BEAM_Dashboard_-_Serotypes_of_concern__Burden_...,61808,.csv,Possible CDC BEAM; requires metadata confirmation,None,,c21d0a7cd5c462fbaf47fc4abc3b103bf36de860b0eaea...
6,BEAM_Dashboard_-_Serotypes_of_concern__Illness...,245837,.csv,Possible CDC BEAM; requires metadata confirmation,None,,72fc2545dd7eafeb4971ed3d6645d1914aaae7215a4e37...
7,BEAM_Dashboard_–_Top_30_Most_Common_Serotypes_...,270714,.csv,Possible CDC BEAM; requires metadata confirmation,None,,d35bcb68f699ffeaf4d94ff22d27ad1070675be29faf86...
8,Dataset_Establishment_Demographic_Data.csv,2879549,.csv,Unrecognized / documentation; do not merge,None,,151e6efc89061ad9c13207b3198e193757967ecdfb7111...
9,Dataset_EstablishmentCategoriesAggregate_20260...,24232,.xlsx,Unrecognized / documentation; do not merge,None,,1c74e0f45888e7f8dca31db351860d21972b8d875487c9...


,inventoried_original_files,total_inventoried_bytes,files_with_recorded_read_errors,scope
0,42,1017948267,0,Recorded original inventory; not a fresh origi...


## 3. Rebuild the dictionary and verify bundled source-table bytes

This invokes the maintained export script in the isolated workspace. It parses the bundled tables, measures null counts, and checks recorded derived-table hashes. It also validates the selected NOAA response checksums. Recorded original-download lineage hashes are carried forward from ingestion manifests; they are not fresh provider authentication.

Code: [dictionary exporter](../scripts/build_data_dictionary.py), [source-specific ingestion adapters](../backend/ingestion/).


In [4]:
run.run_script("scripts/build_data_dictionary.py")
dictionary = pd.read_csv(run.workspace / "reports/data_dictionary.csv")
provenance = run.read_json("reports/source_provenance.json")
display(pd.DataFrame(provenance["datasets"])[[
    "dataset", "provider", "rows", "columns", "grain", "integrity", "time_semantics"
]])
assert (dictionary["null_count"] + dictionary["non_null_count"]).eq(dictionary["rows"]).all()
print("Observed dictionary fields:", len(dictionary))
hash_checks = []
for entry in provenance["datasets"]:
    path = run.workspace / entry["path"]
    if entry.get("sha256") and path.is_file():
        actual = hashlib.sha256(path.read_bytes()).hexdigest()
        hash_checks.append({
            "dataset": entry["dataset"],
            "current_sha256": actual,
            "recorded_sha256": entry["recorded_sha256"],
            "matches": actual == entry["sha256"] == entry["recorded_sha256"],
        })
hash_checks = pd.DataFrame(hash_checks)
assert not hash_checks.empty and hash_checks["matches"].all()
display(hash_checks)


{"datasets": 11, "dictionary_rows": 150, "output_root": "C:\\Users\\ericp\\Downloads\\NAFSI\\reports\\notebook_runs\\00_coverage-8f791e98-f991-4a56-8b1e-b990cc3896a6\\project\\reports"}


,dataset,provider,rows,columns,grain,integrity,time_semantics
0,fsis_poultry,USDA FSIS,78607,9,selected primary raw-poultry sample,matched,collection date; no result-availability date
1,beam_0,CDC BEAM,1650,7,pathogen x HHS region x month,matched,calendar year/month; export snapshot
2,beam_1,CDC BEAM,262616,13,state x month x source type/site x pathogen x ...,matched,calendar year/month; no row publication date
3,beam_2,CDC BEAM,484,13,food category x serotype x provider summary pe...,matched,Year and Year_range have distinct meanings
4,beam_3,CDC BEAM,2412,10,food category x serotype x first-illness year ...,matched,Year_first_ill differs from report Year and Ye...
5,beam_4,CDC BEAM,4818,8,pathogen x serotype/species/subgroup x month,matched,calendar year/month; export snapshot
6,nors_0,CDC NORS,66713,19,reported historical outbreak record,matched,Year/Month; historical export through 2023
7,establishments_0,USDA FSIS MPI directory,175,19,establishment directory record,matched,directory snapshot; grant_date is not a sampli...
8,usda_food_environment_missouri,USDA ERS Food Environment Atlas,115,14,county/county-equivalent,matched,measure-specific 2019/2020/2023; July 2025 wor...
9,integrated_mo_analytic,AEGIS derived retrospective study,1534,26,selected Missouri FSIS sample; origin seven da...,matched,explicit origins and assumed feature lags; see...


Observed dictionary fields: 150


,dataset,current_sha256,recorded_sha256,matches
0,fsis_poultry,12304d0fb04e62c593fa7149eff43874bc2acc724aea49...,12304d0fb04e62c593fa7149eff43874bc2acc724aea49...,True
1,beam_0,a4e91dcab3d677721b1fc1c8d3278802f655fb727d9af3...,a4e91dcab3d677721b1fc1c8d3278802f655fb727d9af3...,True
2,beam_1,f9f997aa85c29152f647f370eff3fe88b757e8bf08810f...,f9f997aa85c29152f647f370eff3fe88b757e8bf08810f...,True
3,beam_2,8984140a2ce478b98560e5711397c5e4bad534547623e6...,8984140a2ce478b98560e5711397c5e4bad534547623e6...,True
4,beam_3,13ef86b755564be3d74a531fbb3c4aeb57b52229febdfd...,13ef86b755564be3d74a531fbb3c4aeb57b52229febdfd...,True
5,beam_4,fa6e75365b49f42b1f210bd44378a4b3e0914211055be1...,fa6e75365b49f42b1f210bd44378a4b3e0914211055be1...,True
6,nors_0,b924774bd85127bb97e4304d6661d3457eba499c99f9ed...,b924774bd85127bb97e4304d6661d3457eba499c99f9ed...,True
7,establishments_0,fa92b0f60d69db0ff54374ddeeec3c7a212d93f433a60c...,fa92b0f60d69db0ff54374ddeeec3c7a212d93f433a60c...,True
8,usda_food_environment_missouri,627e1c413685dd3f69eba64555864ee25a6fa580ebf9c6...,627e1c413685dd3f69eba64555864ee25a6fa580ebf9c6...,True
9,integrated_mo_analytic,ecfd751025d06444049534210a24d38927035af1ec6238...,ecfd751025d06444049534210a24d38927035af1ec6238...,True


In [5]:
display(dictionary.loc[dictionary["null_count"].gt(0), [
    "dataset", "column", "rows", "null_count", "non_null_count", "units", "definition"
]].head(25))
print("These are actual local column null counts. Non-null provider strings may still encode unavailable or invalid analytic values.")
print("No null count in this notebook is converted into an invented zero observation.")


,dataset,column,rows,null_count,non_null_count,units,definition
18,beam_1,State,262616,7,262609,provider state/jurisdiction label,Provider-specific code or name; source units a...
22,beam_1,Serotype/Species,262616,8,262608,provider category,Aggregate serotype/species label; not sequence...
24,beam_1,Outbreak associated isolates,262616,240444,22172,reported isolate count,Provider association field; not evidence linki...
25,beam_1,New multistate outbreaks,262616,251690,10926,reported outbreak count,Provider stratum-specific count; repeated summ...
26,beam_1,New multistate outbreaks - US,262616,261208,1408,reported national outbreak count,National summary may repeat across source dime...
27,beam_1,% Isolates with clinically important antimicro...,262616,259910,2706,percent of provider-defined tested isolates,Provider resistance summary; denominator is no...
28,beam_1,Number of sequenced isolates analyzed by NARMS,262616,252398,10218,reported sequenced isolate count,Provider NARMS analyzed-isolate denominator; i...
38,beam_2,Hospitalization(s),484,254,230,reported hospitalizations,Provider summary; null is not zero.
59,beam_4,% Change,4818,285,4533,percent,Provider change relative to its comparator; mi...
64,nors_0,Etiology,66713,16338,50375,provider category,Reported etiology; string matching and exclusi...


These are actual local column null counts. Non-null provider strings may still encode unavailable or invalid analytic values.
No null count in this notebook is converted into an invented zero observation.


## 4. Validate the historical NOAA snapshot coverage

Returned station/day rows and usable measurements are different denominators. Missing precipitation or quality flags are not filled with zero. The provider date labels and observation attributes remain preserved. Forecast weather and simulated cargo temperature do not enter this historical coverage table.


In [6]:
noaa = run.read_json("reports/noaa_coverage.json")
station_rows, weather_hashes = [], []
for station in noaa["stations"]:
    relative = station["raw_response_path"].replace("\\", "/")
    response = run.workspace / relative
    actual = hashlib.sha256(response.read_bytes()).hexdigest()
    weather_hashes.append({
        "station_id": station["station_id"],
        "current_sha256": actual,
        "recorded_sha256": station["sha256"],
        "matches": actual == station["sha256"],
    })
    for metric, counts in station["metric_coverage"].items():
        station_rows.append({
            "station_id": station["station_id"],
            "city": station["city"],
            "metric": metric,
            "returned_station_days": station["observed_days"],
            "numeric_days": counts["numeric_days"],
            "quality_pass_days": counts["quality_pass_days"],
            "missing_days": counts["missing_days"],
            "flagged_days": counts["flagged_days"],
            "units": counts["units"],
        })
weather_hashes = pd.DataFrame(weather_hashes)
assert weather_hashes["matches"].all()
display(weather_hashes)
display(pd.DataFrame(station_rows))
display(pd.DataFrame([{
    "recorded_status": noaa["status"],
    "observation_start": noaa["start_date"],
    "observation_end": noaa["end_date"],
    "returned_station_days": noaa["observed_station_days"],
    "row_coverage_fraction": noaa["coverage_fraction"],
    "metric_coverage_fraction": noaa["metric_coverage_fraction"],
    "missing_metric_values": noaa["missing_metric_values"],
}]))


,station_id,current_sha256,recorded_sha256,matches
0,USW00003945,9992c1be1348244526e9cf6a01a9063899ec491df7bfc4...,9992c1be1348244526e9cf6a01a9063899ec491df7bfc4...,True
1,USW00003947,9ecfb639b8d4b43acc7a90d76861ba137de0451c06e59b...,9ecfb639b8d4b43acc7a90d76861ba137de0451c06e59b...,True
2,USW00003963,cd0cec6d4c636d9e5ea3454bdcd75d2645bd3fe6b08693...,cd0cec6d4c636d9e5ea3454bdcd75d2645bd3fe6b08693...,True
3,USW00013995,26ca1fac107c2ccb1c9651fe103e1412f94f6670de8103...,26ca1fac107c2ccb1c9651fe103e1412f94f6670de8103...,True
4,USW00013994,3477c320482a7d92d096b45b0d87e234503d8809fc0318...,3477c320482a7d92d096b45b0d87e234503d8809fc0318...,True


,station_id,city,metric,returned_station_days,numeric_days,quality_pass_days,missing_days,flagged_days,units
0,USW00003945,Columbia,TMAX,546,546,546,0,0,degrees C
1,USW00003945,Columbia,TMIN,546,546,546,0,0,degrees C
2,USW00003945,Columbia,PRCP,546,546,546,0,0,mm
3,USW00003947,Kansas City,TMAX,546,546,546,0,0,degrees C
4,USW00003947,Kansas City,TMIN,546,546,546,0,0,degrees C
5,USW00003947,Kansas City,PRCP,546,545,545,1,0,mm
6,USW00003963,Jefferson City,TMAX,546,546,546,0,0,degrees C
7,USW00003963,Jefferson City,TMIN,546,546,546,0,0,degrees C
8,USW00003963,Jefferson City,PRCP,546,539,539,7,0,mm
9,USW00013995,Springfield,TMAX,546,546,546,0,0,degrees C


,recorded_status,observation_start,observation_end,returned_station_days,row_coverage_fraction,metric_coverage_fraction,missing_metric_values
0,partial,2025-01-01,2026-06-30,2730,1.0,0.998901,9


## 5. Read source status without inventing a new API success

The next table displays separately dated connectivity tests. A recorded HTTP success means that request succeeded at its test time; it does not prove a public endpoint works now. This notebook sends no provider request.

The integrated-model coverage below describes the captured research cohort. Actual historical publication dates remain unavailable, assumed lags remain explicit, and operational model activation stays disabled.


In [7]:
api_status = run.read_json("reports/api_status.json")
assert isinstance(api_status, list), "Expected the source-specific recorded status table"
api_table = pd.DataFrame(api_status)
display(api_table.reindex(columns=["id", "tested_at", "success", "http_status", "failure"]))
study = run.read_json("reports/integrated_model_evaluation.json")
display(pd.DataFrame([study["coverage"]]).T.rename(columns={0: "recorded_value"}))
assert study["operational_activation_allowed"] is False
print("Model status:", study["status"])
print("Target:", study["target"])
print("This notebook audits captured coverage; notebook 02 recomputes fitting and evaluation.")


,id,tested_at,success,http_status,failure
0,fsis_recalls,2026-10-31T07:00:26.559932+00:00,True,200.0,NaN
1,fsis_sampling,2026-10-28T06:09:23.318096+00:00,False,NaN,ConnectError: [Errno 11002] getaddrinfo failed
2,fsis_directory,2026-10-28T06:09:23.949178+00:00,False,NaN,ConnectError: [Errno 11002] getaddrinfo failed
3,cdc_beam,2026-10-28T06:09:24.565158+00:00,True,200.0,NaN
4,cdc_nors,2026-10-28T06:09:26.243185+00:00,True,200.0,NaN
5,ncei_docs,2026-10-28T06:09:27.797453+00:00,True,200.0,NaN
6,ncei_historical,2026-10-28T06:09:28.825020+00:00,True,200.0,NaN
7,nws_alerts,2026-10-31T07:00:27.755420+00:00,True,200.0,NaN
8,nws_points,2026-10-28T06:09:30.816909+00:00,True,200.0,NaN
9,openmeteo,2026-10-31T07:00:29.304790+00:00,True,200.0,NaN


,recorded_value
input_fsis_rows,78607
valid_fsis_samples,78553
national_states,46
mo_samples,1534
mo_establishments,16
mo_positives,136
eligible_samples,490
excluded_incomplete_coverage,1044
weather_station_count,5
weather_station_days,2730


Model status: conditional retrospective multi-source feasibility; prospective skill unestablished
Target: Salmonella positivity in a selected Missouri FSIS raw-poultry primary sample collected 7 days after forecast origin
This notebook audits captured coverage; notebook 02 recomputes fitting and evaluation.


## 6. Preserve the source project

The execution manifest records commands, input/output hashes and preservation checks. Results are written to the isolated workspace. Dates, run identifiers and notebook execution metadata will differ on a new run; those are not scientific metric differences.


In [8]:
display(run.verify_preservation())
display(run.finish())


{'passed': True,
 'changed_inputs': [],
 'operator_preserved': True,
 'raw_preserved': True,
 'changed_raw_inputs': []}

{'id': '00_coverage-8f791e98-f991-4a56-8b1e-b990cc3896a6',
 'started_at': '2026-11-02T20:12:36.471309+00:00',
 'finished_at': '2026-11-02T20:12:43.966730+00:00',
 'commands': [{'script': 'scripts/build_data_dictionary.py',
   'arguments': [],
   'exit_code': 0,
   'duration_seconds': 1.17,
   'log': '01-build_data_dictionary.txt',
   'log_sha256': 'b710bb75d3c8a89a3d1a7d88a71e58da45b78f8b4d83dcc1ceb9ae34a217dde6'}],
 'copied_input_count': 282,
 'input_hashes': {'backend/__init__.py': {'exists': True,
   'bytes': 1,
   'sha256': '01ba4719c80b6fe911b091a7c05124b64eeece964e09c058ef8f9805daca546b'},
  'backend/activity.py': {'exists': True,
   'bytes': 17737,
   'sha256': '4532590f947821299df3083c87ad1bf35442027b2ff3cf428ae3417223b32f4d'},
  'backend/adapters.py': {'exists': True,
   'bytes': 7171,
   'sha256': 'd7e02c3f8e3b6815c83f500fad5a39dda48b88e340fc6358feea272e108ee89a'},
  'backend/assistant_intent.py': {'exists': True,
   'bytes': 16307,
   'sha256': '4b8ed2e036d96d71606c07701518b